# DM-NeRF / DM-SR — Final evaluation + meshing (Kaggle)

Use **somente quando `training_state.json` indicar `complete=true`**. Anexe o Notebook Output da sessão final de treinamento em **Add Input → Notebook Output**, ative GPU + Internet e execute **Save Version → Save & Run All**.

Esta sessão não treina. Ela executa avaliação oficial e meshing em FP32 em uma janela separada, evitando que a sessão de 12 h seja consumida pelo treinamento antes das etapas finais.


In [ ]:
from pathlib import Path
import json, shutil, subprocess, sys

def run(*args): subprocess.run([str(x) for x in args],check=True)
run('nvidia-smi')
BASE=Path('/kaggle/temp/dissertacao-dmnerf-finalize')
OUTPUT=Path('/kaggle/working/dmnerf-results')
candidates=[]
for state_path in Path('/kaggle/input').rglob('training_state.json'):
    identity_path=state_path.parent/'identity.json'; ckpt=state_path.parent/'latest.tar'
    if state_path.parent.name!='experiment' or not identity_path.is_file() or not ckpt.is_file(): continue
    state=json.loads(state_path.read_text()); identity=json.loads(identity_path.read_text())
    if identity.get('amp') is True and state.get('complete') is True:
        candidates.append((int(state['iteration']),state_path.parents[3]))
if not candidates:
    raise RuntimeError('Nenhum checkpoint AMP completo encontrado. Termine 200001 passos antes de finalizar.')
iteration,previous=sorted(candidates,key=lambda x:(x[0],str(x[1])))[-1]
print('Checkpoint completo selecionado:',previous,'iteração',iteration)
if OUTPUT.exists(): shutil.rmtree(OUTPUT)
shutil.copytree(previous,OUTPUT)
if BASE.exists(): shutil.rmtree(BASE)
BASE.mkdir(parents=True,exist_ok=True)
REPO=BASE/'sdf-generator'; WORK=BASE/'work'
PIN='e8bef9b7dc7732719c9ded4f0e1024db242c8baa'
run('git','clone','https://github.com/Jfsslemos/sdf-generator.git',REPO)
run('git','-C',REPO,'checkout','--detach',PIN)
assert subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'],text=True).strip()==PIN
run(sys.executable,REPO/'tools/dmnerf/bootstrap.py','--work',WORK,'--lock',OUTPUT/'environment.freeze.txt')
PYTHON=WORK/'env/bin/python'
run(PYTHON,REPO/'tools/dmnerf/prepare.py','--work',WORK)


In [ ]:
run(PYTHON,REPO/'tools/dmnerf/run.py','--work',WORK,'--output',OUTPUT,'--profile','full','--amp','--stages','evaluate','mesh')


In [ ]:
state=json.loads((OUTPUT/'raw/study/experiment/training_state.json').read_text())
print(json.dumps(state,indent=2))
print('Métricas:',OUTPUT/'derived/experiment/metrics.csv')
print('Pacote:',OUTPUT.parent/(OUTPUT.name+'-bundle.zip'))


Envie `dmnerf-results-bundle.zip` ao ChatGPT. Esse bundle será usado para métricas finais, meshing, A0/A1/A2 e preparação do Gazebo.
